# מעבדה 00 — כמה זה מספיק כדי שיתקיים חוק

במעבדה הזו תצפו במספר בר-שחזור מרכיב את עצמו ממספרים בלתי צפויים לחלוטין, תמדדו במדויק כמה
מהר הוא נעשה בר-שחזור, ותפריכו את ההסבר שרוב האנשים פונים אליו ראשון.

אין כאן פיזיקה, וזה בכוונה. לקוביות אין אנרגיה, אין מכל ואין דינמיקה, ולכן מה שגורם לממוצע
שלהן להתייצב אינו יכול להיות מנגנון פיזיקלי — הוא חייב להיות האריתמטיקה של הרבים. האריתמטיקה
הזו היא מה שמודולים 3, 4 ו-8 ישתמשו בו מחדש.

עבדו לפי הסדר. במקום שבו המחברת מבקשת מכם לנבא, כתבו את הניבוי בתא המיועד **לפני** הרצת התא
הבא. זה אינו טקס: ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | $N$ קוביות, כל אחת מראה אחד משישה פנים בעלי הסתברות שווה; הגודל הנמדד הוא הסכום או הממוצע שלהן |
| **דינמיקה** | אין — כל דגימה בלתי תלויה בכל האחרות, בלי זיכרון ביניהן |
| **גבול** | סגור באופן טריוויאלי; $N$ קבוע בניסוי אחד ודבר אינו מוחלף |
| **צבר** | לכל פן הסתברות שווה, מכוח הנחה ולא מכוח הוכחה |
| **מוזנח** | כל הפיזיקה של קוביות אמיתיות — הגלגול, הקפיצה, אי-סימטריה של הנקודות, גרר האוויר |
| **תקף כאשר** | האקראיות מתוארת היטב על ידי דגימות אחידות בלתי תלויות |
| **אופני כשל** | דגימות מתואמות, קוביות מוטות, כל שאלה על הטלה *בודדת* |

כל קוד המודל נמצא ב-`thermolab.sampling` וב-`thermolab.forms` — פתחו וקראו אותם. שום דבר
בקורס הזה אינו מוסתר בתוך תשתית.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package has to be
# installed into the kernel first. Under a local Jupyter it is already importable, so
# this does nothing.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install("thermolab", keep_going=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import forms, sampling
from thermolab.validation import relative_error, scaling_exponent, seed_study

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"one fair die:  mean = {sampling.die_mean():.4f}")
print(f"               variance = {sampling.die_variance():.4f}")
print(f"               sigma = {np.sqrt(sampling.die_variance()):.4f}")
print(f"               sigma/mean = {sampling.die_relative_spread():.4f}  <- the coefficient")

## חלק 1 — צפו בחוק מרכיב את עצמו

שש מאות הטלות של קובייה אחת, לפי הסדר, לצד הממוצע הרץ של אותה סדרה עצמה.

In [ ]:
rolls = sampling.roll_dice(600, rng)
curve = sampling.running_average(rolls)
trials = np.arange(1, rolls.size + 1)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.8))

left.scatter(trials, rolls, s=14, alpha=0.6)
left.axhline(sampling.die_mean(), color="crimson", ls="--")
left.set_yticks(range(1, 7))
left.set_xlabel("roll number")
left.set_ylabel("face shown")
left.set_title("the microscopic reality")

right.plot(trials, curve, lw=1.4)
right.axhline(sampling.die_mean(), color="crimson", ls="--")
right.set_ylim(1, 6)
right.set_xlabel("rolls averaged, N")
right.set_ylabel("average so far")
right.set_title("the same data, accumulated")

plt.tight_layout()
plt.show()

print(f"average of all {rolls.size} rolls: {rolls.mean():.4f}   (exact answer 3.5)")

הפאנל השמאלי לעולם אינו נעשה מסודר. הוא אינו יכול: כל הטלה נדגמת מאותה התפלגות בדיוק כמו
הראשונה, ושום דבר בסדרה אינו משתנה במהלכה.

### נבאו

התא הבא חוזר על הניסוי "מיצוע $N$ קוביות" פעמים רבות, עבור $N = 10$, $100$ ו-$1000$, ומשרטט
נקודה אחת לכל ניסוי שהושלם. לפני שאתם מריצים אותו, התחייבו לתשובה:

- בערך כמה רחוק מ-$3.5$ תשב נקודה טיפוסית עבור $N = 10$? ועבור $N = 1000$?
- בכמה אמור הפיזור האנכי להצטמצם בין שני הפאנלים האלה?

**הניבוי שלכם:**

*(כתבו כאן לפני הרצת התא הבא)*

In [ ]:
sizes = (10, 100, 1000)
n_experiments = 240
columns = [sampling.sample_averages(n, n_experiments, rng) for n in sizes]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, n_per_sample, averages in zip(axes, sizes, columns, strict=True):
    ax.scatter(np.arange(1, n_experiments + 1), averages, s=10, alpha=0.6)
    ax.axhline(sampling.die_mean(), color="crimson", ls="--")
    ax.set_ylim(2.3, 4.7)
    ax.set_title(f"N = {n_per_sample}")
    ax.set_xlabel("experiment number")
axes[0].set_ylabel("average of N dice")
plt.tight_layout()
plt.show()

for n_per_sample, averages in zip(sizes, columns, strict=True):
    print(f"N = {n_per_sample:5d}   measured spread = {averages.std(ddof=1):.4f}"
          f"   predicted = {np.sqrt(sampling.die_variance() / n_per_sample):.4f}")

כל פאנל חולק את אותה סקאלה אנכית, ולכן ההצטמצמות אמיתית ואינה תוצר של גבולות הצירים. גידול פי
מאה ב-$N$ צימצם את הענן פי עשרה — וזהו חוק $N^{-1/2}$, נראה לעין לפני שנגזר.

## חלק 2 — מדדו את המעריך, ואת המקדם

מעריך מותאם לבדו הוא ראיה חלשה: מודלים שגויים רבים מייצרים שיפוע קרוב ל-$-\tfrac{1}{2}$. לכן
אנחנו בודקים גם את המקדם. הגזירה חוזה

$$
\frac{\sigma_{\bar{A}_N}}{\mu_1} = \frac{\sigma_1}{\mu_1} \, N^{-1/2} ,
\qquad \frac{\sigma_1}{\mu_1} \approx 0.488 .
$$

In [ ]:
fit_sizes = np.array([4, 16, 64, 256, 1024])
spreads = np.array([
    sampling.relative_spread_of_average(int(n), n_samples=1500, rng=rng) for n in fit_sizes
])
predicted = np.array([sampling.predicted_relative_spread(int(n)) for n in fit_sizes])

exponent = scaling_exponent(fit_sizes, spreads)
coefficients = spreads * np.sqrt(fit_sizes)

print(f"fitted exponent: {exponent:+.4f}   (predicted -0.5)\n")
print(f"{'N':>6}  {'measured':>10}  {'predicted':>10}  {'rel. error':>10}  {'coeff':>7}")
for n, measured, expected, coefficient in zip(fit_sizes, spreads, predicted, coefficients,
                                              strict=True):
    print(f"{n:6d}  {measured:10.5f}  {expected:10.5f}"
          f"  {relative_error(measured, expected):10.3f}  {coefficient:7.4f}")
print(f"\ncoefficient should approach sigma_1/mu_1 = {sampling.die_relative_spread():.4f}")

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.loglog(fit_sizes, spreads, "o", label="measured")
plt.loglog(fit_sizes, predicted, "-", label=r"$(\sigma_1/\mu_1)\,N^{-1/2}$")
plt.xlabel("N")
plt.ylabel(r"$\sigma_{\bar{A}} / \mu$")
plt.title(f"relative spread of the average (fitted slope {exponent:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

שימו לב שהישר המותאם תואם ב*גובה* ולא רק בשיפוע. זו הטענה החזקה יותר, והיא זו שהייתה נשברת אילו
הגזירה הייתה שגויה באופן שבדיקת שיפוע אינה יכולה לראות.

### הסכום הולך בכיוון ההפוך

אותם נתונים, בקריאה אחרת. התבוננו בפיזור המוחלט של הסכום גדל בעוד הפיזור היחסי של הממוצע מצטמצם.

In [ ]:
sum_spreads = []
average_spreads = []
for n in fit_sizes:
    averages = sampling.sample_averages(int(n), 1500, rng)
    sum_spreads.append(float((n * averages).std(ddof=1)))
    average_spreads.append(float(averages.std(ddof=1) / averages.mean()))

print(f"sum, absolute scatter      fitted exponent {scaling_exponent(fit_sizes, sum_spreads):+.3f}"
      "   (predicted +0.5)")
print(f"average, relative scatter  fitted exponent "
      f"{scaling_exponent(fit_sizes, average_spreads):+.3f}   (predicted -0.5)")

## חלק 3 — דילול, לא פיצוי

הנה הניסוי ששווה להריץ בעצמכם. ההסבר הרגיל לכך שממוצע מתייצב הוא שסטיות מוקדמות מבוטלות על ידי
מאוחרות. אילו זה היה נכון, הרצות ש*התחילו* גבוה במיוחד היו חייבות להמשיך נמוך במיוחד.

אפשר פשוט לבדוק. ייצרו הרצות רבות, השאירו רק את אלה שעשר ההטלות הראשונות שלהן מיצעו מעל $4.5$,
ואז הסתכלו על מה שאותן הרצות עצמן עשו אחר כך.

In [ ]:
n_runs, n_start, n_after = 40_000, 10, 60

starts = rng.integers(1, 7, size=(n_runs, n_start))
afters = rng.integers(1, 7, size=(n_runs, n_after))

hot = starts.mean(axis=1) > 4.5
print(f"{hot.sum()} of {n_runs} runs started hot (first {n_start} rolls averaged > 4.5)")
print(f"  their opening average:                  {starts[hot].mean():.4f}")

after_mean = afters[hot].mean()
after_error = afters[hot].std(ddof=1) / np.sqrt(afters[hot].size)
print(f"  the SAME runs, over the next {n_after} rolls: {after_mean:.4f} +/- {after_error:.4f}")
print(f"  every run, over the next {n_after} rolls:     {afters.mean():.4f}")
print("\nIf the dice compensated, the middle number would sit below 3.5. It does not.")

In [ ]:
# Where the early excess actually goes: it is divided away, never cancelled.
combined = np.hstack([starts[hot], afters[hot]])
mean_curve = np.array([sampling.running_average(run) for run in combined]).mean(axis=0)
k = np.arange(1, combined.shape[1] + 1)

# The comparison curve assumes every roll after the opening block averages exactly 3.5 — no
# correction of any kind — and lets the opening excess be divided away. It is only defined
# once that block is complete, so it starts at k = n_start.
tail = k >= n_start
dilution = (n_start * starts[hot].mean() + (k[tail] - n_start) * sampling.die_mean()) / k[tail]

plt.figure(figsize=(7, 4.2))
plt.plot(k, mean_curve, lw=1.6, label="hot-start runs, mean running average")
plt.plot(k[tail], dilution, "--", lw=1.2, label="pure dilution: excess never corrected")
plt.axhline(3.5, color="crimson", ls=":", label="3.5")
plt.xlabel("rolls averaged, N")
plt.ylabel("average so far")
plt.title("an early excess is divided away, not cancelled")
plt.legend()
plt.tight_layout()
plt.show()

שתי העקומות מונחות זו על זו. הקו המקווקו חושב מתוך ההנחה שכל הטלה אחרי העשירית מתמצעת בדיוק
ל-$3.5$ — בלי שום תיקון — והוא משחזר את המדידה. לקוביות אין זיכרון; העודף המוקדם הוא כמות קבועה
המחולקת ב-$N$ ההולך וגדל.

זו התפיסה השגויה שעליה בנויה שאלת החידון הראשונה של המודול, וכדאי לשים לב שהיא חוזה את *המספר
הנכון* מהסיבה הלא נכונה. צדיקה במעריך אינה אומרת שהמנגנון בידיכם.

## חלק 4 — מדויק ולא מדויק, כמתמטיקה טהורה

החצי השני של האבחון במודול הזה הוא חשבון דיפרנציאלי. לחלק הזה אין דבר עם קוביות; זו פיסת
המתמטיקה שמודול 5 ייבנה עליה, הנפגשת כאן בצורתה הפשוטה ביותר.

תבנית דיפרנציאלית $\omega = M\,\mathrm{d}x + N\,\mathrm{d}y$ הופכת למספר רק אחרי שנותנים לה
מסלול. לפעמים המספר תלוי במסלול ולפעמים לא.

In [ ]:
def route_diagonal(n_points):
    t = np.linspace(0.0, 1.0, n_points)
    return t, t


def route_across_then_up(n_points):
    t = np.linspace(0.0, 1.0, n_points)
    x = np.hstack([t, np.ones(n_points)])
    y = np.hstack([np.zeros(n_points), t])
    return x, y


def route_up_then_across(n_points):
    t = np.linspace(0.0, 1.0, n_points)
    x = np.hstack([np.zeros(n_points), t])
    y = np.hstack([t, np.ones(n_points)])
    return x, y


ROUTES = {
    "diagonal y = x": route_diagonal,
    "across, then up": route_across_then_up,
    "up, then across": route_up_then_across,
}

# omega_1 = y dx + x dy  is d(xy);   omega_2 = y dx  is the differential of nothing.
FORMS = {
    "w1 = y dx + x dy": (lambda x, y: y, lambda x, y: x),
    "w2 = y dx": (lambda x, y: y, lambda x, y: np.zeros_like(x)),
}

print(f"{'':20}" + "".join(f"{name:>18}" for name in ROUTES))
for form_name, (m, n) in FORMS.items():
    values = [forms.line_integral(m, n, *route(801)) for route in ROUTES.values()]
    print(f"{form_name:20}" + "".join(f"{v:18.4f}" for v in values))

In [ ]:
# The criterion decides it without integrating anything at all.
probe_x = np.linspace(0.2, 1.8, 40)
probe_y = np.linspace(0.3, 1.7, 40)

for form_name, (m, n) in FORMS.items():
    gap = forms.mixed_partials_gap(m, n, probe_x, probe_y)
    verdict = "exact" if forms.is_exact(m, n, probe_x, probe_y) else "INEXACT"
    print(f"{form_name:20}  dM/dy - dN/dx = {gap.mean():+.6f}   -> {verdict}")

התבנית הראשונה נותנת $1$ בכל דרך, מפני שהיא $\mathrm{d}(xy)$ והתשובה היא פשוט
$f(1,1) - f(0,0)$. השנייה נותנת $0$, $\tfrac{1}{2}$ או $1$ בהתאם לדרך שבה נסעתם — אותה התחלה,
אותו סוף, שלוש תשובות.

במודול 5 המישור הופך למישור $P$–$V$. התבנית המדויקת מתאינטגרלת לשינוי באנרגיה פנימית, התלוי רק
בנקודות הקצה; הלא מדויקת מתאינטגרלת לעבודה, התלויה בדרך. אין צורך בשום דבר פיזיקלי כדי לראות את
ההבדל — הוא כבר כאן, ב-$y\,\mathrm{d}x$.

## חלק 5 — בדיקות אוטומטיות

כל טענה שלמעלה היא גם מבחן בערכת הבדיקות של הפרויקט. אלה אותן בדיקות, המורצות כאן כדי שמחברת
שמפסיקה בשקט להיות נכונה תיכשל ברעש.

In [ ]:
# 1. The die's exact moments — no approximation, no sampling.
assert sampling.die_mean(6) == 3.5
assert abs(sampling.die_variance(6) - 35 / 12) < 1e-12

# 2. Conservation of counting: every roll lands on a real face.
sample = sampling.roll_dice(5000, np.random.default_rng(1))
assert sample.min() >= 1 and sample.max() <= 6
assert np.bincount(sample, minlength=7).sum() == sample.size

# 3. The running average ends exactly on the plain mean — no drift introduced.
assert abs(sampling.running_average(sample)[-1] - sample.mean()) < 1e-12

# 4. The N^(-1/2) law, with an honest error bar across independent seeds.
study = seed_study(
    lambda r: sampling.relative_spread_of_average(64, 1500, r), n_seeds=8
)
target = sampling.predicted_relative_spread(64)
assert study.agrees_with(target, n_sigma=3.0)

# 5. Path-independence of an exact form, and its failure for an inexact one.
diagonal = forms.line_integral(lambda x, y: y, lambda x, y: x, *route_diagonal(801))
stepped = forms.line_integral(lambda x, y: y, lambda x, y: x, *route_across_then_up(801))
assert abs(diagonal - stepped) < 1e-9
inexact_a = forms.line_integral(lambda x, y: y, lambda x, y: np.zeros_like(x),
                                *route_diagonal(801))
inexact_b = forms.line_integral(lambda x, y: y, lambda x, y: np.zeros_like(x),
                                *route_across_then_up(801))
assert abs(inexact_a - inexact_b) > 0.4

print(f"exact moments           mean {sampling.die_mean():.4f}, "
      f"variance {sampling.die_variance():.6f}")
print(f"relative spread at N=64 {study.mean:.5f} +/- {study.standard_error:.5f}"
      f"  vs predicted {target:.5f}")
print(f"exact form, two routes  {diagonal:.6f} and {stepped:.6f}  (must agree)")
print(f"inexact form, two routes {inexact_a:.6f} and {inexact_b:.6f}  (must differ)")
print("\nall checks passed")

## חלק 6 — חקרו בעצמכם

כווננו את המחוונים ולחצו **Run** כדי לשרטט מחדש. שני דברים שכדאי לעשות:

1. הגדירו את הקובייה לשני פנים (מטבע, המנוקד 1 ו-2). המקדם $\sigma_1/\mu_1$ משתנה; בדקו
   שה*מעריך* אינו משתנה.
2. הגדילו את `n_per_sample` וצפו בענן הנקודות קורס אל הקו המקווקו. אין ערך שבו ההתנהגות משנה את
   אופייה — היא פשוט ממשיכה להצטמצם.

In [ ]:
import ipywidgets as widgets


# interact_manual, not interact: set all three sliders, then press Run. Besides being the
# saner interaction for a callback that redraws two panels, it keeps this notebook fast under
# automated execution — a figure emitted from inside a live `interact` callback costs about
# five minutes per notebook under nbmake on Windows, against four seconds this way.
def explore(n_per_sample=100, n_experiments=200, n_faces=6):
    local = np.random.default_rng(0)
    averages = sampling.sample_averages(n_per_sample, n_experiments, local, n_faces)
    mean = sampling.die_mean(n_faces)
    measured = averages.std(ddof=1) / averages.mean()
    expected = sampling.predicted_relative_spread(n_per_sample, n_faces)

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.scatter(np.arange(1, n_experiments + 1), averages, s=10, alpha=0.6)
    left.axhline(mean, color="crimson", ls="--")
    left.set_xlabel("experiment number")
    left.set_ylabel(f"average of {n_per_sample} draws")
    left.set_title(f"measured spread {measured:.4f} vs predicted {expected:.4f}")

    right.plot(np.arange(1, n_per_sample + 1),
               sampling.running_average(sampling.roll_dice(n_per_sample, local, n_faces)), lw=1.2)
    right.axhline(mean, color="crimson", ls="--")
    right.set_xlabel("rolls averaged, N")
    right.set_ylabel("average so far")
    right.set_title("one run, accumulating")
    plt.tight_layout()
    plt.show()


widgets.interact_manual(
    explore,
    n_per_sample=widgets.IntSlider(min=2, max=2000, step=2, value=100, description="N"),
    n_experiments=widgets.IntSlider(min=50, max=600, step=25, value=200, description="repeats"),
    n_faces=widgets.IntSlider(min=2, max=20, step=1, value=6, description="faces"),
);

## בדקו את הבנתכם

החידון הזה הוא כלי אבחון ולא בחינה. המשוב לכל פריט נוקב במקום שאליו כדאי ללכת אם הוא תפס אתכם.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "00-orientation.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלמטה.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בנימוק שלכם?
2. חלק 3 הפריך הסבר אחד לכך שממוצעים מתייצבים. נסחו במשפט אחד את ההסבר ששרד, בלי להשתמש במילה
   "מתבטל".
3. המחולל שבשימוש כאן מייצר דגימות בלתי תלויות *מעצם בנייתו*. נקבו במסקנה אחת מהיום שלפיכך
   **אינה** מבוססת על ידי המספרים האלה, אף שהכול הסכים.
4. איזו מארבע המיומנויות — יחידות, מוסכמות, נגזרות, הסתברות — עלתה לכם ביוקר היום? מה תעשו בנידון
   לפני מודול 1?

**התשובות שלכם:**

1.
2.
3.
4.